# Phase 2 · Train with the reward (Kaggle, 2× T4)

Compares ways of using the training reward chosen in Phase 1 (**MetricX-24-QE**) on **two model families**, so that findings are not one model's quirk:

| Model | Role | Systems |
|---|---|---|
| `qwen3-1.7b` (Qwen3-1.7B, non-thinking mode) | main | `base`, `opus-mt`, `sft`, `sft+bo8`, `rsft`, `dpo`, `grpo`, `grpo_guarded` |
| `gemma3-1b` (Gemma 3 1B-it) | replication | `base`, `sft`, `dpo`, `grpo`, `grpo_guarded` |

- `sft`: fine-tuned on ~20k News Commentary pairs; the shared starting point.
- `sft+bo8`: best of 8 samples at test time, chosen by MetricX-QE.
- `rsft`: fine-tuned on the best of 8 samples per prompt.
- `dpo`: DPO on (best, worst) sample pairs.
- `grpo`: online RL; the guard checks are only *logged*.
- `grpo_guarded`: the same, with penalties for wrong language, number mismatch, length and copying (the blind spots from Phase 1).

All methods start from the same `sft` model and use the same budget of 5,000 prompts × 8 samples.

**Run it four times:** `qwen3-1.7b` part A, then part B, then `gemma3-1b` part A and part B. Rough times on 2× T4, not yet measured:
- Qwen: part A ~4–5 h, part B ~5–6 h.
- Gemma runs in fp32 on a T4 (it overflows in fp16), so expect it to be slower. Its GRPO stops at the matched budget (1 epoch).

### Setup (once)
1. Kaggle: *Settings* → Accelerator **GPU T4 ×2**, Internet **On**.
2. **`HF_TOKEN` must be a *write* token** (huggingface.co → Settings → Access Tokens). Adapters are saved to a private repo `<your-user>/mtrlhf-en-de-adapters` as they finish, so later parts and re-runs pick them up.
3. Gemma is gated: accept the license at huggingface.co/google/gemma-3-1b-it with the same account.
4. `GH_TOKEN` as in Phase 1: translations, logs and sampled candidates are pushed to `results/p2/<model>/`.
5. For long runs use **Save Version → Save & Run All** so you can close the tab.

Every step skips work that is already saved, so re-running the notebook continues where it stopped.

In [ ]:
# ---- configuration ----
REPO = "https://github.com/guohere/machine-translation-with-RLHF.git"
BRANCH = "claude/compassionate-ritchie-2nga86"
MODEL = "qwen3-1.7b"          # "qwen3-1.7b" (main), then "gemma3-1b" (replication)
PART = "A"                    # "A", then "B" in a second session (or "ALL" on a fast GPU)
HUB_REPO = "auto"             # private HF repo for adapters; None = keep them in this session only
SMOKE_TEST = True             # ~10-minute tiny run of every step first (recommended for each new model)
RUN_GRPO_COMETKIWI = False    # optional extra run: does GRPO on CometKiwi drift into the wrong language?
TEST_SETS = "flores_devtest.en-de wmt24.en-de flores_devtest.de-en flores_devtest.en-zh"
QUICK_SCORES = "chrf metricx24_qe comet22"   # comet22 = the held-out evaluator from Phase 1

In [ ]:
# ---- GPU check and Hugging Face token (works on Kaggle and Colab) ----
import os, subprocess

WORK = next((d for d in ("/kaggle/working", "/content") if os.path.isdir(d)), os.path.expanduser("~"))
REPO_DIR = os.path.join(WORK, "machine-translation-with-RLHF")   # every later cell runs from here
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "NONE - switch the accelerator to a T4 GPU before continuing")

def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        pass
    return os.environ.get(name)

token = get_secret("HF_TOKEN")
if token:
    os.environ["HF_TOKEN"] = token
    print("HF_TOKEN loaded")
else:
    print("WARNING: no HF_TOKEN secret found - the gated CometKiwi model will fail to download")

In [ ]:
# ---- get the code and install (~3 minutes) ----
if not os.path.isdir(REPO_DIR):
    !git clone -q -b {BRANCH} {REPO} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
os.chdir(REPO_DIR)
!git log -1 --format="code version: %h %s"
# transformers stays below 5 so the COMET evaluator works next to TRL (see pyproject.toml)
!pip install -q -e ".[phase1,phase2]"
!pip install -q --no-deps unbabel-comet==2.2.7 "git+https://github.com/google-research/mt-metrics-eval"
!python -c "import torch, transformers, trl, peft, comet; print('ready: torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__, '| GPUs', torch.cuda.device_count())"
os.environ["MTRLHF_HUB_REPO"] = HUB_REPO or "none"

In [ ]:
# ---- per-model paths, GitHub saving, job runner ----
import base64, glob, gzip, shutil, subprocess, sys
os.chdir(REPO_DIR)  # independent of where earlier cells left the working directory
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from mtrlhf.models import CheckpointStore, model_paths

P = model_paths(MODEL)
SYSTEMS = P["systems"]
RES = f"results/p2/{MODEL}"
store = CheckpointStore.from_env(P["ckpt"], HUB_REPO, P["hub_prefix"])
print(f"model {MODEL} = {P['model_id']} | systems: {', '.join(SYSTEMS)}")
print("adapter store:", f"{store.hub_repo}/{P['hub_prefix']}" if store.hub_repo else "local only (this session)")

GH_TOKEN = get_secret("GH_TOKEN")
print("GH_TOKEN loaded" if GH_TOKEN else "No GH_TOKEN: results stay in this session only")

def _git(*args):
    auth = base64.b64encode(f"x-access-token:{GH_TOKEN}".encode()).decode()
    cmd = ["git", "-c", "user.name=Phase 2 notebook", "-c", "user.email=noreply@users.noreply.github.com",
           "-c", f"http.https://github.com/.extraheader=AUTHORIZATION: basic {auth}", *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).replace(GH_TOKEN, "***").replace(auth, "***")

def save_results(message):
    """Copy small artifacts into results/p2/ and push them to the branch."""
    os.makedirs(RES, exist_ok=True)
    if os.path.exists("data/p2/data_stats.json"):
        shutil.copy("data/p2/data_stats.json", "results/p2/")
    if os.path.exists(f"{P['model_data']}/pairs_stats.json"):
        shutil.copy(f"{P['model_data']}/pairs_stats.json", RES)
    if os.path.exists(f"{P['model_data']}/candidates.jsonl"):
        with open(f"{P['model_data']}/candidates.jsonl", "rb") as src, gzip.open(f"{RES}/candidates.jsonl.gz", "wb") as dst:
            shutil.copyfileobj(src, dst)
    for run_dir in glob.glob(f"{P['runs']}/*"):
        for f in ("log_history.json", "samples.jsonl"):
            if os.path.exists(f"{run_dir}/{f}"):
                os.makedirs(f"{RES}/training/{os.path.basename(run_dir)}", exist_ok=True)
                shutil.copy(f"{run_dir}/{f}", f"{RES}/training/{os.path.basename(run_dir)}/")
    if not GH_TOKEN:
        return
    _git("add", "results/p2")
    if _git("diff", "--cached", "--quiet")[0] != 0:
        code, out = _git("commit", "-q", "-m", message)
        if code:
            print("commit failed:", out); return
    code, out = _git("pull", "-q", "--rebase", "origin", BRANCH)
    if code:
        print("pull failed:", out); return
    code, out = _git("push", "-q", "origin", f"HEAD:{BRANCH}")
    print(f"saved to GitHub: {message}" if code == 0 else f"push failed: {out}")

def restore_candidates():
    if os.path.exists(f"{RES}/candidates.jsonl.gz") and not os.path.exists(f"{P['model_data']}/candidates.jsonl"):
        os.makedirs(P["model_data"], exist_ok=True)
        with gzip.open(f"{RES}/candidates.jsonl.gz", "rb") as src, open(f"{P['model_data']}/candidates.jsonl", "wb") as dst:
            shutil.copyfileobj(src, dst)
        print("restored sampled candidates from an earlier run")

def have(name):
    return name in ("base", "opus-mt") or store.has(name)

import time

def n_gpus():
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
    except FileNotFoundError:
        return 0
    return sum(line.startswith("GPU") for line in out.splitlines())

def run_jobs(jobs, report_every=600):
    """Run (name, shell command) jobs, two at a time on two GPUs if available, else one after another.
    Output goes to <name>.log; the last log line is printed every 10 minutes."""
    slots = max(1, min(n_gpus(), 2))
    pending, running, last = list(jobs), [], time.time()
    while pending or running:
        while pending and len(running) < slots:
            name, cmd = pending.pop(0)
            gpu = min(set(range(slots)) - {r[2] for r in running})
            log = open(f"{name}.log", "w")
            proc = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT,
                                    env={**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)})
            running.append((name, proc, gpu, log))
            print(f"started {name} on GPU {gpu}", flush=True)
        time.sleep(15)
        for r in list(running):
            if r[1].poll() is not None:
                r[3].close()
                running.remove(r)
                tail = open(f"{r[0]}.log").read().strip().splitlines()[-3:]
                print(f"{r[0]} finished (exit code {r[1].returncode}):", *tail, sep="\n  ", flush=True)
        if running and time.time() - last > report_every:
            last = time.time()
            for name, *_ in running:
                lines = open(f"{name}.log").read().strip().splitlines()
                print(f"[{time.strftime('%H:%M')}] {name}: {lines[-1][-160:] if lines else '(starting)'}", flush=True)

### Step 1 · Data and test sets (CPU, ~5–10 min, shared by both models)
News Commentary en-de → 20k SFT pairs, 500 validation pairs, and 5,000 English prompts (disjoint). FLORES-200 dev/devtest (en-de, de-en, en-zh) and WMT24 en-de. Training sentences that occur in any test set are removed.

In [ ]:
if not os.path.exists("data/p2/prompts.jsonl"):
    !python scripts/p2_prepare_data.py --out data/p2
!cat data/p2/data_stats.json
save_results("Phase 2: data statistics")

### Smoke test (~10–15 min; once per model)
Runs SFT, sampling, GRPO and translation for a few steps with the **real** model and the real MetricX reward. It checks GPU memory, logins, the chat template and library versions before the long runs. It writes to `ckpt/<model>_debug/` and never touches the real results or the Hub.

**Check the printed prompt:** for Qwen3 it must end with an empty `<think>\n\n</think>` block (non-thinking mode), and the sample translations should be German.

In [ ]:
if SMOKE_TEST:
    from mtrlhf.data.prompts import build_messages
    from mtrlhf.models import load_tokenizer
    tok = load_tokenizer(P["model_id"])
    print(repr(tok.apply_chat_template(build_messages("The weather is nice today."), tokenize=False, add_generation_prompt=True)))
    dbg = f"{P['ckpt']}_debug"
    !python scripts/p2_train.py --model {MODEL} --config configs/p2/sft.yaml --debug
    !python scripts/p2_sample.py --model {MODEL} --ckpt {dbg} --system sft --limit 8 --n 4 --out {P['runs']}_debug/candidates.jsonl
    !python -c "import json; r=[json.loads(l) for l in open('{P['runs']}_debug/candidates.jsonl')]; print(*[s['hyp'] for s in r[0]['samples']], sep='\n')"
    !python scripts/p2_train.py --model {MODEL} --config configs/p2/grpo_guarded.yaml --debug
    !python scripts/p2_translate.py --model {MODEL} --ckpt {dbg} --systems sft --testsets flores_devtest.en-de --limit 8 --out {P['runs']}_debug/outputs --score chrf

### Part A · Step 2 · Supervised fine-tuning → `sft` (~45–60 min for Qwen; longer for Gemma)

In [ ]:
if PART in ("A", "ALL"):
    !python scripts/p2_train.py --model {MODEL} --config configs/p2/sft.yaml
    save_results(f"Phase 2 ({MODEL}): sft")

### Part A · Step 3 · Sample 8 translations per prompt and score them with MetricX-QE (~60–75 min)
The shared budget: 5,000 prompts × 8 samples = 40,000 reward calls. The policy runs on GPU 0 and MetricX on GPU 1. Resumable.

In [ ]:
if PART in ("A", "ALL"):
    restore_candidates()
    !python scripts/p2_sample.py --model {MODEL} --system sft --n 8 --reward metricx24_qe
    !python scripts/p2_build_pairs.py --model {MODEL}
    save_results(f"Phase 2 ({MODEL}): sampled candidates and pairs")

### Part A · Step 4 · RSFT and DPO (~30–45 min; side by side on two GPUs)

In [ ]:
if PART in ("A", "ALL"):
    jobs = [(m, f"python scripts/p2_train.py --model {MODEL} --config configs/p2/{m}.yaml")
            for m in ("rsft", "dpo") if m in SYSTEMS]
    run_jobs(jobs)
    save_results(f"Phase 2 ({MODEL}): {' and '.join(m for m, _ in jobs)}")

### Part B · Step 5 · GRPO and guarded GRPO (side by side on two GPUs)
Both start from `sft`. Every 125 steps the adapter is saved as `<name>-step<k>`. **Step 625** (= 5,000 prompts × 8 samples) is the matched-budget model used in the comparison. Qwen continues to step 1250 only to trace the over-optimization curve, while Gemma stops at 625. If the session stops after step 625, the main result is already saved and the run is skipped next time.

In the logs, `rewards/metricx24_qe/mean` should rise. The `guard_*` lines are the exploit rates (logged in both runs, penalized only in `grpo_guarded`).

In [ ]:
BUDGET_STEP = 625

def grpo_job(name):
    if store.has(name) or store.has(f"{name}-step{BUDGET_STEP}"):
        print(f"{name}: matched-budget checkpoint already saved, skipping")
        return []
    return [(name, f"python scripts/p2_train.py --model {MODEL} --config configs/p2/{name}.yaml --force")]

if PART in ("B", "ALL"):
    runs = ["grpo", "grpo_guarded"] + (["grpo_cometkiwi"] if RUN_GRPO_COMETKIWI else [])
    run_jobs([job for name in runs for job in grpo_job(name)])
    save_results(f"Phase 2 ({MODEL}): GRPO runs")

### Step 6 · Translate the test sets (~1 h per part)
Greedy decoding for every system, plus best-of-8 for `sft` (main model only). In Part B, also every saved GRPO checkpoint on FLORES dev, for the over-optimization curve. Results go to `results/p2/<model>/outputs/` and are pushed after each block.

In [ ]:
T = f"python scripts/p2_translate.py --model {MODEL}"
if PART in ("A", "ALL"):
    systems = " ".join(s for s in ["base", "opus-mt", "sft", "rsft", "dpo"] if s in SYSTEMS and have(s))
    !{T} --systems {systems} --testsets {TEST_SETS} flores_dev.en-de
    save_results(f"Phase 2 ({MODEL}): translations ({systems})")
    if "sft+bo8" in SYSTEMS:
        !{T} --systems sft --best-of 8 --testsets flores_devtest.en-de wmt24.en-de flores_dev.en-de
        save_results(f"Phase 2 ({MODEL}): translations (sft + best-of-8)")

if PART in ("B", "ALL"):
    for run in ["grpo", "grpo_guarded"] + (["grpo_cometkiwi"] if RUN_GRPO_COMETKIWI else []):
        main = f"{run}-step{BUDGET_STEP}"
        if have(main):
            !{T} --systems {main} --testsets {TEST_SETS} flores_dev.en-de
        curve = " ".join(f"{run}-step{k}" for k in range(125, 1251, 125) if k != BUDGET_STEP and have(f"{run}-step{k}"))
        if curve:
            !{T} --systems {curve} --testsets flores_dev.en-de
        save_results(f"Phase 2 ({MODEL}): translations ({run})")

### Step 7 · Quick evaluation (~15–30 min)
Mean chrF, MetricX-QE (the training reward) and COMET-22 (held out) per system and test set, plus how often each guard fires. This is a first look; Phase 3 adds significance tests, the over-optimization plots and the hacking analysis.

In [ ]:
!python scripts/p2_translate.py --model {MODEL} --summary-only --score {QUICK_SCORES} > /dev/null
save_results(f"Phase 2 ({MODEL}): quick evaluation (part {PART})")
from IPython.display import Markdown, display
display(Markdown(open(f"{RES}/outputs/quick_eval.md").read()))

### Troubleshooting
| Symptom | Fix |
|---|---|
| `Hub mirror disabled` | `HF_TOKEN` is read-only. Create a *write* token, or run Parts A and B in the same session |
| `401` / gated repo for `gemma-3-1b-it` | Accept the Gemma license on Hugging Face with the account of `HF_TOKEN` |
| Qwen3 prompt has no `</think>` block, or outputs start with `<think>` | Tell Claude: the non-thinking switch in `src/mtrlhf/models.py` did not match this tokenizer |
| `CUDA out of memory` in GRPO | Add `--set batch_size=2 grad_accum=32` to the GRPO command (same 8 prompts per step) |
| `CUDA out of memory` in sampling | Add `--batch-size 32` to the `p2_sample.py` command |
| A step was interrupted | Run the notebook again: finished adapters, candidates and translations are skipped |
| `push failed` | Check the `GH_TOKEN` secret (fine-grained, this repository, *Contents: Read and write*) |
| FLORES download fails | The notebook falls back to the gated `openlanguagedata/flores_plus`; accept its terms on Hugging Face |